In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Parameter yang umum digunakan untuk transpilasi

<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami menyarankan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Halaman ini menjelaskan beberapa parameter yang paling umum digunakan untuk transpilasi lokal. Parameter-parameter ini dikonfigurasi menggunakan argumen ke [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) atau [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<span id="approx-degree"></span>
## Tingkat aproksimasi
Kamu bisa menggunakan tingkat aproksimasi untuk menentukan seberapa dekat kamu ingin Circuit hasil transpilasi cocok dengan Circuit yang diinginkan (input). Ini adalah nilai float dalam rentang (0.0 - 1.0), di mana 0.0 adalah aproksimasi maksimum dan 1.0 (default) berarti tidak ada aproksimasi. Nilai yang lebih kecil menukar akurasi output dengan kemudahan eksekusi (yaitu, lebih sedikit Gate). Nilai defaultnya adalah 1.0.

Dalam sintesis uniter dua-Qubit (digunakan pada tahap awal semua level dan untuk tahap optimasi dengan optimization level 3), nilai ini menentukan target fidelitas dari output dekomposisi. Artinya, seberapa besar kesalahan yang dimasukkan ketika representasi matriks sebuah Circuit dikonversi menjadi gate diskrit. Jika tingkat aproksimasi memiliki nilai lebih rendah (lebih banyak aproksimasi), Circuit output dari sintesis akan lebih berbeda dari matriks input, tetapi kemungkinan juga memiliki lebih sedikit Gate (karena operasi dua-Qubit arbitrari apapun dapat didekomposisi secara sempurna dengan paling banyak tiga CX Gate) dan lebih mudah dijalankan.

Ketika tingkat aproksimasi kurang dari 1.0, Circuit dengan satu atau dua CX Gate mungkin akan disintesis, yang menghasilkan lebih sedikit kesalahan dari hardware, tetapi lebih banyak dari aproksimasi. Karena CX adalah Gate yang paling mahal dalam hal kesalahan, mungkin menguntungkan untuk mengurangi jumlahnya dengan mengorbankan fidelitas dalam sintesis (teknik ini digunakan untuk meningkatkan quantum volume pada perangkat IBM&reg;: [Validating quantum computers using randomized model circuits](https://arxiv.org/abs/1811.12926)).

Sebagai contoh, kita membuat `UnitaryGate` dua-Qubit acak yang akan disintesis pada tahap awal. Mengatur `approximation_degree` kurang dari 1.0 mungkin menghasilkan Circuit aproksimasi. Kita juga harus menentukan `basis_gates` untuk memberi tahu metode sintesis Gate mana yang dapat digunakan untuk sintesis aproksimasi.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

Ini menghasilkan output `2` karena aproksimasi membutuhkan lebih sedikit CX Gate.

<span id="seed"></span>
## Seed generator angka acak
Beberapa bagian dari Transpiler bersifat stokastik, sehingga transpilasi yang diulang mungkin menghasilkan hasil yang berbeda. Untuk mendapatkan hasil yang dapat direproduksi, kamu bisa mengatur seed untuk generator angka pseudoacak menggunakan argumen `seed_transpiler`. Eksekusi berulang menggunakan seed yang sama akan menghasilkan hasil yang sama.

Contoh:

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## Layout awal
Sebelum transpilasi, Qubit yang terdapat dalam Circuit kamu adalah Qubit virtual yang tidak selalu berkorespondensi dengan Qubit fisik pada Backend target. Kamu bisa menentukan pemetaan awal Qubit virtual ke Qubit fisik menggunakan argumen `initial_layout`. Perlu diingat bahwa layout Qubit akhir mungkin berbeda dari layout awal karena Transpiler mungkin memindahkan Qubit menggunakan swap gate atau cara lain.

Dalam contoh di bawah ini, kita membuat layout awal untuk Backend mock [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) dengan membuat objek [`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout). Layout kita memetakan Qubit pertama Circuit ke Qubit 5 Sherbrooke, dan memetakan Qubit kedua Circuit ke Qubit 6 Sherbrooke. Perlu diingat bahwa Qubit fisik selalu direpresentasikan dengan bilangan bulat.

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

Selain menentukan objek Layout, kamu juga bisa melewatkan daftar bilangan bulat, di mana elemen ke-$i$ dari daftar berisi Qubit fisik yang harus dipetakan oleh Qubit ke-$i$. Misalnya:

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

Kamu bisa menggunakan fungsi [`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map) untuk membuat diagram grafik perangkat dengan informasi kesalahan dan dengan label Qubit fisik. Kamu juga bisa melihat diagram serupa di halaman [Compute resources](https://quantum.cloud.ibm.com/computers).